In [0]:
from pyspark.ml import pipelines as dp

In [0]:
taxis.find_all_taxis().show(5)

In [0]:
#Yellow Trip Read
trip_input_df = (spark.read
      .format("csv")
      .option("header", True)
      .load("/databricks-datasets/nyctaxi/tripdata/yellow/yellow_tripdata_2019-01.csv.gz"))

display(trip_input_df)

In [0]:
from pyspark.sql.functions import col, substring, to_date, regexp_replace

trip_df = (trip_input_df
    .withColumn("year_month", regexp_replace(substring("tpep_pickup_datetime",1,7), '-', '_'))
    .withColumn("pickup_dt", to_date("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss")) 
    .withColumn("dropoff_dt", to_date("tpep_dropoff_datetime", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("tip_pct", col("tip_amount") / col("total_amount"))
    .limit(1000)
)
display(trip_df)

In [0]:
#Zone Read
zone_df = (spark.read
  .option("header", "true")
  .csv("/databricks-datasets/nyctaxi/taxizone/taxi_zone_lookup.csv")
)

zone_df.show()

In [0]:

full_df = (trip_df
  .join(zone_df,
        trip_df.PULocationID == zone_df.LocationID,
        how="left")
  .drop("LocationID")
)

In [0]:
(full_df.write
   .format("delta")
   .mode("overwrite")
   .save("/tmp/datasets/datakickstart/yellow_trips_sample")
)

In [0]:
full_df.write.mode("overwrite").saveAsTable("yellow_trips_sample")

In [0]:
new_df = spark.read.table("yellow_trips_sample")

## Alternate Read & Write (meant for local environment)
If you need this data local, you can download just one month of Yellow trip data plus the Taxi Zone Lookup Table. Data can be found at https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

Bash commands to download two files:
```bash
mkdir -p /tmp/datasets/nyctaxi/taxizone
cd /tmp/datasets/nyctaxi/taxizone
wget https://d37ci6vzurychx.cloudfront.net/misc/taxi+_zone_lookup.csv
mv taxi+_zone_lookup.csv taxi_zone_lookup.csv

mkdir -p /tmp/datasets/nyctaxi/tables/nyctaxi_yellow
cd /tmp/datasets/nyctaxi/tables/nyctaxi_yellow
wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet
```

In [0]:
#Yellow Trip Read

# trip_format = "delta"
trip_format = "parquet"

trip_input_df = (spark.read
      .format(trip_format)
      .load("/tmp/datasets/nyctaxi/tables/nyctaxi_yellow"))

trip_input_df.show()

In [0]:
#Zone Read
zone_df = (spark.read
  .option("header", "true")
  .csv("/tmp/datasets/nyctaxi/taxizone/taxi_zone_lookup.csv")
)

zone_df.show()

In [0]:
from pyspark.sql.functions import col, substring, to_date, regexp_replace

trip_df = (trip_input_df
    .withColumn("year_month", regexp_replace(substring("tpep_pickup_datetime",1,7), '-', '_'))
    .withColumn("pickup_dt", to_date("tpep_pickup_datetime", "yyyy-MM-dd HH:mm:ss")) 
    .withColumn("dropoff_dt", to_date("tpep_dropoff_datetime", "yyyy-MM-dd HH:mm:ss"))
    .withColumn("tip_pct", col("tip_amount") / col("total_amount"))
    .limit(1000)
)


In [0]:
full_df = (trip_df
  .join(zone_df,
        trip_df.PULocationID == zone_df.LocationID,
        how="left")
  .drop("LocationID")
)

In [0]:
(full_df.write
    .format("parquet")
    .save("/tmp/datasets/datakickstart/yellow_trips_sample_parquet")
)